# ai03 Walkthrough — INSTRUCTOR SOLUTIONS
**DO NOT DISTRIBUTE TO STUDENTS**

## Machine Learning with Decision Trees
### Complete end-to-end walkthrough using Titanic dataset

This notebook covers:
- **Lesson 03a**: Loading, exploring, and cleaning data
- **Lesson 03b**: Building, evaluating, and visualizing decision tree models

# PART 1: LOADING, EXPLORING, AND CLEANING DATA (Lesson 03a)

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import accuracy_score, classification_report
import matplotlib.pyplot as plt

print("✓ All libraries imported successfully!")

In [ ]:
titanicData = pd.read_csv('Titanic Dataset.csv')
print("✓ Dataset loaded successfully!")
print(f"Dataset shape: {titanicData.shape[0]} rows, {titanicData.shape[1]} columns")

In [ ]:
print(titanicData.head())

In [ ]:
print(titanicData.tail())

In [ ]:
print(titanicData.info())

In [ ]:
print(titanicData.describe())

In [ ]:
columnsToKeep = ['pclass', 'survived', 'sex', 'age', 'sibsp', 'parch', 'fare', 'embarked']
titanicData = titanicData[columnsToKeep]

print("✓ Selected relevant columns")
print(f"Dataset shape after column selection: {titanicData.shape}")
print(f"Columns: {list(titanicData.columns)}")

In [ ]:
print("\nMissing values BEFORE cleaning:")
print(titanicData.isnull().sum())
print(f"\nTotal missing values: {titanicData.isnull().sum().sum()}")

In [ ]:
rowsBeforeDrop = len(titanicData)
titanicData = titanicData.dropna()
rowsAfterDrop = len(titanicData)

print(f"✓ Dropped rows with missing values")
print(f"Rows before: {rowsBeforeDrop}")
print(f"Rows after: {rowsAfterDrop}")
print(f"Rows removed: {rowsBeforeDrop - rowsAfterDrop}")
print(f"\nDataset shape after cleaning: {titanicData.shape}")

In [ ]:
print("Before encoding:")
print(titanicData[['sex', 'embarked']].head())

titanicData = pd.get_dummies(titanicData, columns=['sex', 'embarked'], drop_first=True)

print("\nAfter encoding:")
print(titanicData.head())
print(f"\nNew columns created: {list(titanicData.columns)}")

# PART 2: BUILDING, EVALUATING, AND VISUALIZING MODELS (Lesson 03b)

In [ ]:
X = titanicData.drop('survived', axis=1)
y = titanicData['survived']

print("✓ Separated features and target")
print(f"Features (X) shape: {X.shape}")
print(f"Features (X) columns: {list(X.columns)}")
print(f"\nTarget (y) shape: {y.shape}")
print(f"Target (y) values: {y.unique()}")
print(f"\nTarget distribution:")
print(y.value_counts())

In [ ]:
xTrain, xTest, yTrain, yTest = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

print("✓ Data split into train/test sets")
print(f"Training set size: {xTrain.shape[0]} rows ({len(xTrain)/len(X)*100:.1f}%)")
print(f"Testing set size: {xTest.shape[0]} rows ({len(xTest)/len(X)*100:.1f}%)")
print(f"\nTotal rows used: {len(xTrain) + len(xTest)}")

In [ ]:
model = DecisionTreeClassifier(
    max_depth=4,
    random_state=42
)

model.fit(xTrain, yTrain)

print("✓ Decision tree model trained successfully!")
print(f"Model type: {type(model).__name__}")
print(f"Max depth: {model.max_depth}")
print(f"Number of features: {model.n_features_in_}")
print(f"Feature names: {X.columns.tolist()}")

In [ ]:
predictions = model.predict(xTest)

print("✓ Predictions made!")
print(f"Number of predictions: {len(predictions)}")
print(f"Unique prediction values: {set(predictions)}")
print(f"\nFirst 20 predictions: {predictions[:20]}")
print(f"\nHow many predicted to survive? {sum(predictions)}")
print(f"How many predicted to NOT survive? {len(predictions) - sum(predictions)}")

In [ ]:
accuracy = accuracy_score(yTest, predictions)

print("="*80)
print("MODEL EVALUATION")
print("="*80)
print(f"Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")
print(f"\nInterpretation:")
print(f"  - Out of {len(yTest)} test passengers")
print(f"  - Model correctly predicted {int(accuracy*len(yTest))} survivors")
print(f"  - Model made {len(yTest) - int(accuracy*len(yTest))} mistakes")

In [ ]:
print("\n" + "="*80)
print("CLASSIFICATION REPORT")
print("="*80)
print(classification_report(yTest, predictions, target_names=['Did Not Survive', 'Survived']))

In [ ]:
plt.figure(figsize=(20, 10))
plot_tree(
    model,
    feature_names=X.columns,
    class_names=['Did Not Survive', 'Survived'],
    filled=True,
    rounded=True,
    fontsize=10
)
plt.title("Decision Tree for Titanic Survival Prediction\n(max_depth=4)", fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

print("✓ Tree visualization complete!")

In [ ]:
importances = model.feature_importances_

featureImportance = list(zip(X.columns, importances))
featureImportance.sort(key=lambda x: x[1], reverse=True)

print("="*80)
print("FEATURE IMPORTANCE")
print("="*80)
for feature, importance in featureImportance:
    bar = '█' * int(importance * 50)
    print(f"{feature:15} | {importance:6.4f} | {bar}")

print(f"\nTotal importance: {sum(importances):.4f}")

In [ ]:
newPassenger1 = pd.DataFrame({
    'pclass': [1],
    'age': [25],
    'sibsp': [1],
    'parch': [0],
    'fare': [100],
    'sex_male': [0],
    'embarked_Q': [0],
    'embarked_S': [0]
})

newPassenger2 = pd.DataFrame({
    'pclass': [3],
    'age': [30],
    'sibsp': [0],
    'parch': [0],
    'fare': [7],
    'sex_male': [1],
    'embarked_Q': [0],
    'embarked_S': [1]
})

pred1 = model.predict(newPassenger1)[0]
pred2 = model.predict(newPassenger2)[0]

prob1 = model.predict_proba(newPassenger1)[0]
prob2 = model.predict_proba(newPassenger2)[0]

print("="*80)
print("PREDICTIONS FOR NEW PASSENGERS")
print("="*80)
print("\nPassenger 1: 1st class female, age 25, $100 fare")
print(f"  Prediction: {'SURVIVED ✓' if pred1 == 1 else 'DID NOT SURVIVE ✗'}")
print(f"  Confidence: {prob1[pred1]*100:.2f}%")

print("\nPassenger 2: 3rd class male, age 30, $7 fare")
print(f"  Prediction: {'SURVIVED ✓' if pred2 == 1 else 'DID NOT SURVIVE ✗'}")
print(f"  Confidence: {prob2[pred2]*100:.2f}%")

In [ ]:
max_depths = [2, 3, 4, 5, 6, 7, 8]
results = []

for depth in max_depths:
    tempModel = DecisionTreeClassifier(max_depth=depth, random_state=42)
    tempModel.fit(xTrain, yTrain)
    
    trainPred = tempModel.predict(xTrain)
    testPred = tempModel.predict(xTest)
    
    trainAcc = accuracy_score(yTrain, trainPred)
    testAcc = accuracy_score(yTest, testPred)
    
    results.append({
        'max_depth': depth,
        'train_accuracy': trainAcc,
        'test_accuracy': testAcc
    })
    
    print(f"max_depth={depth}: Train={trainAcc:.4f}, Test={testAcc:.4f}")

print("\n" + "="*80)
print("INTERPRETATION")
print("="*80)
print("Notice:")
print("- As max_depth increases, training accuracy increases")
print("- Test accuracy increases then decreases (overfitting!)")
print("- Best max_depth balances train and test accuracy")